# Exploring Deltares 1D networks with pyramids-mesh

Rivers and channels are not surfaces. A hydraulic model of a stream does not cover the ground with triangles or
quads the way a 2D flexible mesh does -- it describes the watercourse as a **1D network**: a set of nodes joined by
edges, routing flow along reaches. There are no faces at all.

[Deltares D-Flow FM](https://www.deltares.nl/en/software/modules/d-flow-flexible-mesh) stores such a network in a
UGRID NetCDF file under a `mesh1d` topology. pyramids keeps a 1D network in its own dedicated class, `Mesh1d` --
separate from the 2D `MeshDataset`/`Mesh2d` world -- and the `deltares1d` driver reads a D-Flow FM `.nc` straight
into it. This notebook tours a handful of **real** D-Flow FM networks: a clean one with explicit node coordinates,
two whose geometry is stored as *branch + offset* (and reconstructed for you), a combined 1D2D model, and a 2D-only
control file that the 1D driver correctly refuses.

## Setup

We import `Mesh1d` -- the class a 1D network comes back as -- alongside `MeshDataset`, the 2D wrapper every *other*
pyramids-mesh driver returns. Keeping both in view makes the split explicit: a 1D network is read with
`Mesh1d.read_file`, not `MeshDataset.read_file`. Importing from `pyramids` also puts its vendored GDAL/PROJ data in
place, so nothing here is import-order sensitive -- just import the names you use.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

from pyramids.netcdf.ugrid import Mesh1d, MeshDataset

## Locate the sample files

This notebook lives in `docs/examples/real-data/`, so the repository's `examples/data/real-data/` folder is
**three** levels up. Paths inside a notebook are written relative to the notebook itself, and pyramids' `read_file`
takes a `pathlib.Path` directly -- we pass the `Path`, never `str(path)`. Every network for this driver sits in the
`deltares1d/` subfolder.

In [ ]:
DATA = Path("../../../examples/data/real-data")
DELTARES = DATA / "deltares1d"
net_file = DELTARES / "xy-explicit_1d_readable__korte-woerden.nc"
net_file.name, net_file.exists()

## Why `MeshDataset.read_file` refuses

Before reading the network the right way, it is worth seeing the wrong way fail. pyramids' general UGRID reader
always tries to build a **2D** mesh: it looks for a `face_node_connectivity` table. A pure-1D network has no faces,
so the reader raises. We catch the error so the notebook keeps running -- this is exactly why 1D needs its own
`Mesh1d.read_file` entry point.

In [ ]:
try:
    MeshDataset.read_file(net_file)
except Exception as e:
    print(type(e).__name__, "->", e)

The reader asked for a face table (`face_node_connectivity`) and there was none -- a network is edges and nodes
only. That is the signal to switch to the 1D path.

## Read the network

`Mesh1d.read_file` with `driver="deltares1d"` parses the UGRID `mesh1d` topology -- node coordinates and the
edge-node table -- straight into a `Mesh1d`. No manual array handling, no external NetCDF library.

In [ ]:
net = Mesh1d.read_file(net_file, driver="deltares1d")
{"n_node": net.n_node, "n_edge": net.n_edge,
 "crs": net.crs,
 "bounds": tuple(round(b, 1) for b in net.bounds)}

Immediately after the first read, we let cleopatra pick the matplotlib backend. In a notebook it selects the
inline backend so figures embed in the cell output; in a plain script or CI it falls back to headless `Agg`. We do
not force a backend or use `%matplotlib inline`.

In [ ]:
from cleopatra.config import Config

Config.set_matplotlib_backend()

86 nodes, 91 edges, CRS **EPSG:28992** (Amersfoort / RD New, metres), spanning roughly 320 m x 685 m. There are
**more edges than nodes**, which means the network is not a simple tree -- it contains at least one **loop**. A tree
of `n` nodes has exactly `n - 1` edges; every edge beyond that closes a cycle.

## Edge-length distribution

Edge lengths are a natural property of a network graph -- the along-channel length of each reach segment. Because
the CRS is projected (metres), the numbers are in metres. We summarise all 91 with pandas.

In [ ]:
lengths = pd.Series(net.edge_lengths, name="edge_length_m")
lengths.describe().round(2).to_frame()

Lengths run from a fraction of a metre up to ~218 m, averaging ~30 m -- the channel has been cut into many short
computational segments, with a few long ones. The tiny minimum is typical where two reaches meet at a junction.

## Draw the network by hand

A `Mesh1d` has **no** `plot()` of its own -- it is a graph, not a raster or a filled surface. So we draw it
ourselves: build one line segment per edge from its two node coordinates, collect them into a `LineCollection`, then
scatter the nodes on top. Equal aspect keeps the geometry true (the CRS is in metres), and -- *only because this
network carries a CRS* -- we drape a basemap with cleopatra's `add_tiles`.

In [ ]:
from cleopatra.basemap.tiles import add_tiles

node_x, node_y = net.node_x, net.node_y
enc = net.edge_node_connectivity.data           # (n_edge, 2) node indices
segments = [[(node_x[a], node_y[a]), (node_x[b], node_y[b])] for a, b in enc]

fig, ax = plt.subplots(figsize=(6, 9))
lc = LineCollection(segments, colors="#1a6faf", linewidths=2.0, alpha=0.9, zorder=2)
ax.add_collection(lc)
ax.plot(node_x, node_y, "o", color="white", ms=5, mec="#1a6faf", mew=1.2, zorder=3)
pad = 30
ax.set_xlim(node_x.min() - pad, node_x.max() + pad)
ax.set_ylim(node_y.min() - pad, node_y.max() + pad)
ax.set_aspect("equal")
ax.set_title("korte-woerden - 1D channel network", fontsize=11, pad=8)
ax.axis("off")
if net.crs:
    add_tiles(ax, crs=net.crs, alpha=0.55)

There is the channel schematic on its real-world location: a branching, partly looped network of reaches. Each
white dot is a network node; each blue segment is an edge the solver routes flow along.

We will read three more networks below, so we wrap that same recipe into a small helper and reuse it.

In [ ]:
def draw_network(net, title):
    """Draw a Mesh1d as edges (LineCollection) + nodes (scatter), with a basemap if it has a CRS."""
    nx, ny = net.node_x, net.node_y
    segs = [[(nx[a], ny[a]), (nx[b], ny[b])] for a, b in net.edge_node_connectivity.data]
    fig, ax = plt.subplots(figsize=(6, 7))
    ax.add_collection(LineCollection(segs, colors="#1a6faf", linewidths=1.5, alpha=0.9, zorder=2))
    ax.plot(nx, ny, "o", color="white", ms=3, mec="#1a6faf", mew=0.8, zorder=3)
    pad = 0.03 * max(np.ptp(nx), np.ptp(ny))
    ax.set_xlim(nx.min() - pad, nx.max() + pad)
    ax.set_ylim(ny.min() - pad, ny.max() + pad)
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=11, pad=8)
    ax.axis("off")
    if net.crs:
        add_tiles(ax, crs=net.crs, alpha=0.55)
    return ax

## The network as vectors

`to_geodataframe()` turns the network into a geopandas `GeoDataFrame`: one **LineString** per edge
(`location="edge"`) or one **Point** per node (`location="node"`), CRS-stamped from the network. That is the bridge
to the whole vector-GIS ecosystem -- spatial joins, export to shapefile/GeoPackage, overlay with other layers.

In [ ]:
gdf_edges = net.to_geodataframe(location="edge")
gdf_edges.head()

Each row is a LineString. geopandas draws it directly -- the same channel network, now as a proper vector
layer:

In [ ]:
ax = gdf_edges.plot(figsize=(6, 8), color="steelblue", lw=1)
ax.set_aspect("equal")
if net.crs:
    add_tiles(ax, crs=net.crs)

The node view gives one Point per node. Both frames carry the EPSG:28992 CRS, so they are ready to overlay or
export:

In [ ]:
gdf_nodes = net.to_geodataframe(location="node")
{"edge_rows": len(gdf_edges), "node_rows": len(gdf_nodes),
 "crs": str(gdf_edges.crs)}

## Branch + offset vs explicit x/y

korte-woerden stored its node coordinates explicitly. But D-Flow FM more often stores 1D geometry *implicitly*: a
node is given as a **branch id** plus an **offset** (a distance measured along that branch), and the x/y must be
reconstructed by walking the branch geometry. The `deltares1d` driver does that reconstruction for you (via
pyramids-core's `place_nodes_on_network`), so a branch+offset file reads into exactly the same `Mesh1d` -- you get
real coordinates either way.

Here is the `magdalena` network, whose x/y are reconstructed from branch+offset:

In [ ]:
mag = Mesh1d.read_file(DELTARES / "branchoffset_1d_hidden-xy__magdalena.nc", driver="deltares1d")
{"n_node": mag.n_node, "n_edge": mag.n_edge, "crs": mag.crs,
 "node_x_has_nan": bool(np.isnan(mag.node_x).any()),
 "bounds": tuple(round(b, 1) for b in mag.bounds)}

447 nodes and 446 edges -- exactly `n - 1`, so this one is a pure **tree** (no loops), a single branching river
system. The coordinates came back fully populated (no NaNs) even though the file never stored them explicitly: that
is the branch+offset reconstruction at work. This file carries **no CRS** (`crs` is `None`), so `draw_network` plots
it in bare model coordinates with no basemap -- which our `if net.crs:` guard handles automatically.

In [ ]:
draw_network(mag, "magdalena - reconstructed from branch + offset")

The Magdalena schematic: a large dendritic river network, drawn purely from coordinates the driver rebuilt out
of branch ids and offsets.

## The 1D2D case

Some D-Flow FM models are **combined 1D2D**: a 1D channel network coupled to a 2D floodplain mesh in the same file.
`Mesh1d.read_file` with the `deltares1d` driver pulls out just the **1D** part -- the network -- and leaves the 2D
mesh to the 2D readers. Here is the 1D network inside the `moergestels` 1D2D model:

In [ ]:
moer = Mesh1d.read_file(DELTARES / "branchoffset_1d2d_links_no-hidden-xy__moergestels.nc", driver="deltares1d")
{"n_node": moer.n_node, "n_edge": moer.n_edge, "crs": moer.crs,
 "bounds": tuple(round(b, 1) for b in moer.bounds)}

296 nodes, 295 edges -- again a tree, the drainage network of the model. The companion 2D floodplain mesh in the
same file is simply not part of this object; the 1D driver returns the network and nothing else.

In [ ]:
draw_network(moer, "moergestels - 1D network from a 1D2D model")

## A 2D-only control: when the driver correctly refuses

Finally, the negative case. `mesh2d-only_no-mesh1d_control__nofaces.nc` carries a **2D** mesh topology and *no* 1D
network at all. Asking the `deltares1d` driver for a 1D network from it must fail -- there is nothing 1D to read --
and it does, with a clear message. We catch it.

In [ ]:
control = DELTARES / "mesh2d-only_no-mesh1d_control__nofaces.nc"
try:
    Mesh1d.read_file(control, driver="deltares1d")
except Exception as e:
    print(type(e).__name__, "->", e)

The driver reports *no 1D `mesh_topology`* and declines -- it refuses to guess. This file is a 2D-only control
(a mesh of faces, not a network of reaches); it belongs to the 2D `MeshDataset` world, not to `Mesh1d`. The two
topology dimensions have separate, non-overlapping readers, and that separation is the whole point.

## Takeaway

A 1D network is a **graph**, not a surface. pyramids keeps it in a dedicated `Mesh1d`, and the `deltares1d` driver
reads a D-Flow FM UGRID `.nc` straight into it -- `Mesh1d.read_file(path, driver="deltares1d")` -- whether the node
geometry is stored explicitly or as branch+offset, and whether the file is pure-1D or a combined 1D2D model. From a
`Mesh1d` you get node coordinates, the edge-node table, edge lengths, bounds, a CRS when the file carries one, and a
`GeoDataFrame` of lines or points for the vector-GIS world. There are **no faces, areas or centroids** -- and that
is correct: a channel network has none. Reach for `MeshDataset` only when the file actually describes a 2D surface,
as the control file showed.